# 03 — Data integrity (teardown T1–T3)

Predictions for every experiment here were committed before it ran
(`analysis/PREDICTIONS.md`); deviations are in `analysis/DEVIATIONS.md`. All multi-seed
numbers come from `scripts/run_teardown.py` → `results/teardown.json`.

In [1]:
import json
import numpy as np
import pandas as pd
from heart_audit.data import PROJECT_ROOT, TARGET, FEATURES, load_kaggle918, load_uci920, load_statlog270, to_kaggle_schema
from heart_audit.provenance import match_sources, filled_cells, slope_disagreements
from heart_audit.plots import fill_rule

R = json.loads((PROJECT_ROOT / "results" / "teardown.json").read_text(encoding="utf-8"))
kaggle, uci = load_kaggle918(), to_kaggle_schema(load_uci920())
matches = match_sources(kaggle, uci)
cells = filled_cells(kaggle, uci, matches)
def verdict_table(result, keys):
    rows = []
    for k in keys:
        v = result[k]
        if isinstance(v, list):
            rows += [{"prediction": f"{k} ({g['column']})", "verdict": g["verdict"]} for g in v]
        else:
            rows.append({"prediction": k, "verdict": v["verdict"]})
    return pd.DataFrame(rows)

## Where the published CSV came from

The dataset author documents `303 + 294 + 123 + 200 + 270 (Statlog) = 1190`,
`duplicates 272`, `final 918`. Statlog is a re-coded copy of Cleveland, and the four UCI files
hold two exact duplicates, so the published CSV should be the four UCI files minus two rows.

In [2]:
statlog = load_statlog270()
key = [f for f in FEATURES if f != "ST_Slope"]
statlog_in_cleveland = statlog[key].merge(uci.loc[uci.source == "cleveland", key].drop_duplicates(), on=key, how="left", indicator=True)
print("Statlog rows matching a Cleveland row:", (statlog_in_cleveland._merge == "both").sum(), "of", len(statlog))
print("exact duplicates inside UCI's four files:", load_uci920().drop(columns="source").duplicated().sum())
print("UCI rows - 2 =", len(uci) - 2, "| published rows =", len(kaggle))
print("missing cells: UCI", int(uci[FEATURES].isna().sum().sum()), "| published", int(kaggle[FEATURES].isna().sum().sum()))

Statlog rows matching a Cleveland row: 270 of 270
exact duplicates inside UCI's four files: 2
UCI rows - 2 = 918 | published rows = 918
missing cells: UCI 662 | published 0


The published CSV has no missing values at all. Every UCI `?` became a concrete value.
Matching each UCI row on only the fields it recorded pairs published rows one-to-one with UCI
rows. ST_Slope is excluded from the match, and agreement on it validates the pairing.

In [3]:
paired = matches.uci_row.notna()
print(f"one-to-one pairs: {paired.sum()} of {len(kaggle)}")
print(f"ST_Slope agreement where UCI recorded it: {(paired & matches.uci_slope.notna()).sum()} rows, {slope_disagreements(matches)} disagreements")
cells.column.value_counts().rename("filled cells recovered").to_frame()

one-to-one pairs: 912 of 918
ST_Slope agreement where UCI recorded it: 610 rows, 0 disagreements


,filled cells recovered
column,
ST_Slope,302
FastingBS,89
Oldpeak,60
RestingBP,57
MaxHR,53
ExerciseAngina,53
Cholesterol,28
RestingECG,2


## T1 — The filled values were decided by the diagnosis

In [4]:
tables = []
for col in ["ST_Slope", "FastingBS", "ExerciseAngina"]:
    rows = cells[cells.column == col]
    t = pd.crosstab(rows.kaggle_value.astype(str), kaggle[TARGET].iloc[rows.kaggle_row].to_numpy())
    t.columns = ["no heart disease", "heart disease"]
    t.index = [f"{col} filled as {v}" for v in t.index]
    tables.append(t)
pd.concat(tables)

,no heart disease,heart disease
ST_Slope filled as Flat,0,111
ST_Slope filled as Up,191,0
FastingBS filled as 0,14,0
FastingBS filled as 1,0,75
ExerciseAngina filled as N,20,0
ExerciseAngina filled as Y,0,33


In [5]:
for col in ["Oldpeak", "Cholesterol", "MaxHR", "RestingBP"]:
    rows = cells[cells.column == col]
    y = kaggle[TARGET].iloc[rows.kaggle_row].to_numpy()
    v = rows.kaggle_value.astype(float).to_numpy()
    print(f"{col:12s} filled mean: no disease {v[y == 0].mean():7.2f} (n={(y == 0).sum():2d}) | disease {v[y == 1].mean():7.2f} (n={(y == 1).sum():2d})"
          f" | distinct values shared by both classes: {len(set(v[y == 0]) & set(v[y == 1]))}")

Oldpeak      filled mean: no disease    0.20 (n=21) | disease    1.39 (n=39) | distinct values shared by both classes: 0
Cholesterol  filled mean: no disease  210.18 (n=17) | disease  155.45 (n=11) | distinct values shared by both classes: 0
MaxHR        filled mean: no disease  136.05 (n=20) | disease  119.21 (n=33) | distinct values shared by both classes: 3
RestingBP    filled mean: no disease  134.40 (n=20) | disease  137.41 (n=37) | distinct values shared by both classes: 9


Every filled ST_Slope is `Flat` for a patient with heart disease and `Up` for a patient
without (302 of 302). FastingBS (89/89) and ExerciseAngina (53/53) follow the same rule, and
filled Oldpeak and Cholesterol values share no value between the classes. This shows *what*
the filled values do. It does not identify the procedure that produced them.

In [6]:
panels = []
for g in R["P1.1"]:
    a, b = g["a"], g["b"]
    col = g["column"]
    in_scope = matches.uci_row.notna() & matches.source.isin(["hungary", "va"])
    filled = np.zeros(len(kaggle), bool); filled[cells.loc[cells.column == col, "kaggle_row"]] = True
    rate = lambda m, v: kaggle.loc[m & (kaggle[col] == v), TARGET].mean()
    panels.append({"title": f"{col} (Hungary + VA)", "values": [a, b],
                   "filled": [rate(in_scope & filled, a), rate(in_scope & filled, b)],
                   "observed": [rate(in_scope & ~filled, a), rate(in_scope & ~filled, b)],
                   "n_filled": g["n_filled"], "n_observed": g["n_observed"]})
fill_rule(panels, PROJECT_ROOT / "images" / "fill_rule.png")

WindowsPath('C:/Users/ethan/Desktop/Coding Projects/heart-disease-audit/images/fill_rule.png')

![](../images/fill_rule.png)

In [7]:
pd.DataFrame([{k: g[k] for k in ("column", "gap_filled", "gap_observed", "difference", "ci95", "n_filled", "n_observed", "small_group", "verdict")}
              for g in R["P1.1"]])

,column,gap_filled,gap_observed,difference,ci95,n_filled,n_observed,small_group,verdict
0,ST_Slope,1.0,0.341270,0.658730,"[0.4631869223504886, 0.8587201870976705]","[95, 191]","[144, 28]",False,supported
1,FastingBS,1.0,0.300071,0.699929,"[0.5976322458612264, 0.7992796416752252]","[8, 7]","[87, 386]",True,supported


FastingBS's filled groups in Hungary + VA are small (8 and 7 rows), so that panel is flagged.
Switzerland supplies most filled FastingBS values (74), and they follow the same rule
(table above).

### What the fill is worth (P1.3)

Both arms impute honestly inside the pipeline from training rows. The only difference is
whether the 644 recovered filled cells keep the author's values or are treated as missing.

In [8]:
t1 = pd.read_csv(PROJECT_ROOT / "results" / "teardown" / "t1_seeds.csv")
d = t1.reverted - t1.published
print(f"median RF accuracy with the published values: {t1.published.median():.4f}; with them reverted: {t1.reverted.median():.4f}")
print(f"median paired difference {np.median(d):+.4f}, central 95% [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]")
R["P1.3"]

median RF accuracy with the published values: 0.8641; with them reverted: 0.7935
median paired difference -0.0707, central 95% [-0.1250, -0.0217]


{'median_paired_diff': -0.07065217391304357,
 'central95': [-0.125, -0.021739130434782594],
 'mean_paired_diff': -0.07225543478260871,
 'cells_reverted': 644,
 'verdict': 'supported'}

### Removing the rows (P1.2, the spec's prediction 4)

Removing the 302 rows whose ST_Slope was filled lowers median RF accuracy from the control's
level. The pre-registered null is uninterpretable (see `analysis/DEVIATIONS.md`): matching
outcome counts removes 191 of the 218 unfilled healthy rows. The healthy class that remains
is then mostly rows whose slope was filled by the label, which is why null accuracy is about
0.94. Only the full-vs-removed comparison is interpreted.

In [9]:
R['P1.2']

{'rows_removed': 302,
 'median_full': 0.8695652173913043,
 'median_filled_removed': 0.8145161290322581,
 'null_medians_central95': [0.9354838709677419, 0.9516129032258065],
 'excess_over_majority_full': 0.3152173913043478,
 'excess_over_majority_filled_removed': 0.1693548387096775,
 'excess_over_majority_null_median': 0.29032258064516137,
 'verdict': 'supported'}

## T2 — Provenance as a predictor

In [10]:
pd.DataFrame({"P2.1 (source only)": R["P2.1"], "P2.2 (missingness only)": R["P2.2"]}).T

,in_sample_auc,cv_auc,site_majority_accuracy,verdict,missingness_only_cv_auc,kaggle_chol_zero_auc
P2.1 (source only),0.719633,0.69735,0.669565,falsified,NaN,NaN
P2.2 (missingness only),NaN,NaN,NaN,supported,0.687644,0.625216


P2.1 is **falsified** as registered: the cross-validated AUC of hospital alone (0.697) is 0.022
below its in-sample value (0.720), just outside the 0.02 margin. Hospital alone still predicts
the label well above chance, and so do missingness indicators alone (P2.2).

## T3 — Duplicates (counterfactual)

In [11]:
t3 = pd.read_csv(PROJECT_ROOT / "results" / "teardown" / "t3_seeds.csv")
print(f"1,190-row merge, random split: median RF {t3.random.median():.4f}; duplicate-aware split: {t3.grouped.median():.4f}")
print(f"test rows with a duplicate in training (random split): median {t3.contamination.median():.1%}")
R["P3.1"]

1,190-row merge, random split: median RF 0.8697; duplicate-aware split: 0.7895
test rows with a duplicate in training (random split): median 36.6%


{'rows': 1190,
 'duplicates': 272,
 'median_random': 0.8697478991596639,
 'median_grouped': 0.7894736842105263,
 'contamination_median': 0.36554621848739494,
 'verdict': 'supported'}

This shows what leaving Statlog in would have done. **The published CSV does not have this
defect**: its author removed all 272 duplicates.

## Verdicts

In [12]:
verdict_table(R, ['P1.1', 'P1.2', 'P1.3', 'P2.1', 'P2.2', 'P3.1'])

,prediction,verdict
0,P1.1 (ST_Slope),supported
1,P1.1 (FastingBS),supported
2,P1.2,supported
3,P1.3,supported
4,P2.1,falsified
5,P2.2,supported
6,P3.1,supported
